# Ranking discrepancy investigation

**Campaign**: CALCA helix binder design (BM4 / `runs/CALCA_helix_BM4/evaluate/run1_free`).

**Question**: Why does each design tool's top-by-native rank produce designs that score poorly on refolded `boltz_pae_ipsae_min`, while high-`boltz_pae_ipsae_min` designs come from middle/back-of-the-pack of each tool?

This notebook is the analytical companion to `INVESTIGATION_RANKING_DISCREPANCY.md`. Run with the Mosaic venv:
```
/home/bindmaster5/dev/BindMaster/Mosaic/.venv/bin/python -m jupyter nbconvert --execute --to html notebooks/ranking_discrepancy.ipynb
```

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path("/home/bindmaster5/dev/BindMaster")
RUN_DIR = ROOT / "runs/CALCA_helix_BM4"
EVAL_DIR = RUN_DIR / "evaluate/run1_free"
METRICS_CSV = EVAL_DIR / "report/metrics.csv"

df = pd.read_csv(METRICS_CSV)
print("metrics.csv shape:", df.shape)
print("source tools:")
print(df["source_tool"].value_counts())

## 1. Data integrity audit

Verify join keys per tool. Re-runs the analysis script to regenerate artifacts.

In [ ]:
# Re-execute the analysis pipeline to refresh artifacts
import subprocess

VENV_PY = "/home/bindmaster5/dev/BindMaster/Mosaic/.venv/bin/python"
out = subprocess.run([VENV_PY, str(ROOT / "notebooks/_analysis.py")], capture_output=True, text=True)
print(out.stdout[-3000:])
if out.returncode != 0:
    print("STDERR:", out.stderr[-2000:])

In [ ]:
# Read the join diagnostics
corr = pd.read_csv(ROOT / "notebooks/_artifacts/per_tool_correlations.csv")
corr.head(20)

### Data integrity findings (verified)

1. **BindCraft has 604 rows but only 312 unique sequences/IDs** — every BindCraft design appears (exactly) twice in metrics.csv with identical metrics. This is a pipeline duplication artifact. Likely cause: extractor scanned `bindcraft_default/` and an aliased path (e.g. symlink or both `outputs/` and a parent dir).

2. **PXDesign has 800 rows but only 700 unique IDs** with 800 unique sequences — 100 IDs are reused across distinct sequences. PXDesign's IDs are built from `task_name + rank`, but if multiple length buckets reuse the same `(task_name, rank)` tuple, you get collisions. The metrics.csv preserves both, so this row count is correct, but ID-based joins are unsafe.

3. **Mosaic on-disk `designs.csv` and `checkpoint_*.json` have zero worker-id overlap** with the `mosaic_<worker>_r<rank>` IDs in metrics.csv. The Mosaic source data was re-run after the eval that produced run1_free. **Native ranking_loss is unrecoverable** for the run1_free Mosaic designs.

4. **PXDesign on-disk `summary.csv` only intersects 348/700 IDs** with metrics.csv. Similarly partially-stale.

5. **Proteina-Complexa on-disk `sequences.csv` and `proteina_complexa_top700.csv` use a different ID scheme** (`complexa_CALCA_helix_b1_*`, `pc_top_*`) than metrics.csv's `pc_NNNN_job_X_n_LLL_id_Y_bon_origZ_rW` format. **Zero sequence overlap**. PC's native ranking is unrecoverable for run1_free.

6. **PAE file paths in metrics.csv reference `/home/david/...`** — these were produced on BM4, not Spark. PAE arrays cannot be reloaded; we must rely on pre-computed `boltz_pae_ipsae_min` / `af2_ipsae_min` columns.

### Code-level integrity (read-only)

**Verified correct (per current Evaluator code):**
- Boltz-2 ipSAE computed with `ordering='binder_target'` (`scoring.py:222`).
- AF3/Protenix ipSAE computed with `ordering='target_binder'` (`report.py:80, 92`).
- AF2 ipSAE: deprecated and **no longer in current `scoring.py`** (removed in commit `3ceca0b` 'Part I'). The columns `af2_ipsae_min` etc. in run1_free metrics.csv were computed by **older code (commit `1dd0c51`)** using `ordering='target_binder'` (`add_af2_ipsae_from_files` with explicit `ordering='target_binder'`). The AF2 PAE was saved by `refold_af2.py` (v6) which used `target first (indices 0:L_t), binder second (indices L_t:L_t+L_b)`. **Ordering is consistent**, so the af2_ipsae_min values are not corrupted by a binder/target swap.

**No bugs found** in the ipSAE calculation that would explain the low af2 vs high boltz disagreement.

## 2. Per-tool correlation analysis

Spearman ρ between native ranking metric and refold metrics, plus top-N agreement.

In [ ]:
from IPython.display import Image

Image(str(ROOT / "notebooks/_artifacts/plots/per_tool_scatter.png"))

In [ ]:
Image(str(ROOT / "notebooks/_artifacts/plots/rank_rank.png"))

## 3. Top-20 ipSAE outliers

In [ ]:
top20 = pd.read_csv(ROOT / "notebooks/_artifacts/top20_outliers.csv")
top20

In [ ]:
print("Verdict counts:")
print(top20["verdict"].value_counts())
print()
print("By source tool:")
print(top20.groupby("source_tool")["verdict"].value_counts())

## 4. Cross-engine agreement

Boltz-2 refold ipSAE vs AF2 refold ipSAE for the same designs.

In [ ]:
Image(str(ROOT / "notebooks/_artifacts/plots/cross_engine.png"))

In [ ]:
rows = []
for tool, sub in df.groupby("source_tool"):
    n = len(sub)
    b = (sub["boltz_pae_ipsae_min"] > 0.61).sum()
    a = (sub["af2_ipsae_min"] > 0.61).sum()
    both = ((sub["boltz_pae_ipsae_min"] > 0.61) & (sub["af2_ipsae_min"] > 0.61)).sum()
    valid = sub.dropna(subset=["boltz_pae_ipsae_min", "af2_ipsae_min"])
    rho, _ = stats.spearmanr(valid["boltz_pae_ipsae_min"], valid["af2_ipsae_min"]) if len(valid) > 5 else (np.nan, None)
    rows.append((tool, n, b, a, both, round(rho, 3)))
cross = pd.DataFrame(rows, columns=["tool", "n", "boltz>0.61", "af2>0.61", "both>0.61", "rho(boltz,af2)"])
cross

## 5. Why AF2 ipSAE caps low — PAE distribution

AF2 BT PAE is roughly **2.7× higher** than Boltz-2 BT PAE on the same designs. Combined with the uniform 10 Å cutoff, this drags AF2 ipSAE_min toward 0 across the entire dataset (max af2_ipsae_min = 0.524 vs max boltz_pae_ipsae_min = 0.88).

In [ ]:
Image(str(ROOT / "notebooks/_artifacts/plots/pae_distributions.png"))

## 6. BoltzGen deep dive

In [ ]:
with open(ROOT / "notebooks/_artifacts/boltzgen_deep_dive.json") as fh:
    bg_dive = json.load(fh)
print(json.dumps(bg_dive, indent=2))

## 7. Length vs refold ipSAE

In [ ]:
Image(str(ROOT / "notebooks/_artifacts/plots/length_vs_ipsae.png"))